## Model Size Interaction & Vulnerability Scaling

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ANALYSIS_DIR = Path.cwd()
PLOTS_DIR = ANALYSIS_DIR / "plots"
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

recursive_color = "#B52B27"       # red
human_control_color = "#014C8C"   # blue

condition_colors = {
    "Recursive": recursive_color,
    "Human_Control": human_control_color
}

rec_colors = {
    42: "#D9534F",
    123: "#8C1B17"
}

ctrl_colors = {
    42: "#0275D8",
    123: "#01325C"
}

df_int = pd.read_csv("model_size_interaction.csv")

plt.style.use(
    "seaborn-v0_8-whitegrid"
    if "seaborn-v0_8-whitegrid" in plt.style.available
    else "default"
)

# Plot 1: Bounded Performance & Faithfulness Metrics
fig1, ax1 = plt.subplots(figsize=(9.5, 5.5))

bounded = [
    "context_supported_rate",
    "disease_f1",
    "chemical_f1",
    "cosine_similarity",
    "rouge_l",
    "bertscore_f1",
]
df_b = (
    df_int[df_int["metric"].isin(bounded)]
    .copy()
    .sort_values(by="3B_minus_0.5B")
)

y_pos = np.arange(len(df_b))
width = 0.35

ax1.barh(
    y_pos - width / 2,
    df_b["Qwen2.5-0.5B"],
    width,
    label="Qwen2.5-0.5B",
    color=human_control_color,
    alpha=0.9,
)
ax1.barh(
    y_pos + width / 2,
    df_b["Qwen2.5-3B"],
    width,
    label="Qwen2.5-3B",
    color=recursive_color,
    alpha=0.9,
)

ax1.set_yticks(y_pos)
ax1.set_yticklabels(
    [m.replace("_", " ").title() for m in df_b["metric"]], fontsize=11
)
ax1.axvline(0, color="black", linestyle="--", linewidth=0.9, alpha=0.7)
ax1.set_xlabel(
    "Net Recursive Effect (Recursive Δ - Control Δ)",
    fontsize=11,
    fontweight="bold",
)
ax1.set_title(
    "Cross-Model Vulnerability: Alignment, Similarity & Entity Extraction",
    fontsize=12,
    fontweight="bold",
)
ax1.legend(loc="upper left", frameon=True)

for i, (_, row) in enumerate(df_b.iterrows()):
  diff = row["3B_minus_0.5B"]
  x_val = min(row["Qwen2.5-0.5B"], row["Qwen2.5-3B"]) - 0.012
  ax1.text(
      x_val,
      i,
      f"3B Δ: {diff:+.3f}",
      va="center",
      ha="right",
      fontsize=9.5,
      fontweight="bold",
      color="#333333",
  )

ax1.set_xlim(left=ax1.get_xlim()[0] - 0.08)
plt.tight_layout()
plt.savefig(
    PLOTS_DIR / "model_size_interaction_bounded.png",
    dpi=300,
    bbox_inches="tight",
)
plt.show()

# Plot 2: Scale Metrics (Perplexity and Word Count)
fig2, ax2 = plt.subplots(figsize=(7.5, 4.5))

df_s = (
    df_int[df_int["metric"].isin(["perplexity", "word_count"])]
    .copy()
    .sort_values(by="3B_minus_0.5B")
)
y_pos2 = np.arange(len(df_s))

ax2.barh(
    y_pos2 - width / 2,
    df_s["Qwen2.5-0.5B"],
    width,
    label="Qwen2.5-0.5B",
    color=human_control_color,
    alpha=0.9,
)
ax2.barh(
    y_pos2 + width / 2,
    df_s["Qwen2.5-3B"],
    width,
    label="Qwen2.5-3B",
    color=recursive_color,
    alpha=0.9,
)

ax2.set_yticks(y_pos2)
ax2.set_yticklabels(
    [m.replace("_", " ").title() for m in df_s["metric"]], fontsize=11
)
ax2.axvline(0, color="black", linestyle="--", linewidth=0.9, alpha=0.7)
ax2.set_xlabel("Net Change (Units)", fontsize=11, fontweight="bold")
ax2.set_title(
    "Cross-Model Vulnerability: Text Length & Perplexity",
    fontsize=12,
    fontweight="bold",
)
ax2.legend(loc="lower right", frameon=True)

for i, (_, row) in enumerate(df_s.iterrows()):
  diff = row["3B_minus_0.5B"]
  max_val = max(row["Qwen2.5-0.5B"], row["Qwen2.5-3B"])
  ax2.text(
      max_val + 0.8,
      i,
      f"3B Δ: {diff:+.2f}",
      va="center",
      ha="left",
      fontsize=9.5,
      fontweight="bold",
      color="#333333",
  )

ax2.set_xlim(right=ax2.get_xlim()[1] + 5)
plt.tight_layout()
plt.savefig(
    PLOTS_DIR / "model_size_interaction_scale.png", dpi=300, bbox_inches="tight"
)
plt.show()

## Priority 2 Forest Plot

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ANALYSIS_DIR = Path.cwd()
PLOTS_DIR = ANALYSIS_DIR / "plots"
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

recursive_color = "#B52B27"       # red
human_control_color = "#014C8C"   # blue

condition_colors = {
    "Recursive": recursive_color,
    "Human_Control": human_control_color
}

rec_colors = {
    42: "#D9534F",
    123: "#8C1B17"
}

ctrl_colors = {
    42: "#0275D8",
    123: "#01325C"
}

df_within = pd.read_csv("within_condition_inference_g0_g3.csv")

plt.style.use(
    "seaborn-v0_8-whitegrid"
    if "seaborn-v0_8-whitegrid" in plt.style.available
    else "default"
)

fig, axes = plt.subplots(1, 2, figsize=(14, 6), sharey=True)
models = ["Qwen2.5-0.5B", "Qwen2.5-3B"]
metrics = [
    "Disease F1",
    "Chemical F1",
    "ROUGE-L",
    "BERTScore F1",
    "Cosine similarity",
]

for ax, model in zip(axes, models):
  sub = df_within[
      (df_within["model"] == model) & (df_within["metric"].isin(metrics))
  ]
  agg = (
      sub.groupby(["condition", "metric"])[
          ["mean_change", "ci_95_low", "ci_95_high"]
      ]
      .mean()
      .reset_index()
  )

  y_idx = np.arange(len(metrics))
  offset = 0.16

  rec = agg[agg["condition"] == "Recursive"].set_index("metric").loc[metrics]
  x_rec = rec["mean_change"].values
  err_rec = [x_rec - rec["ci_95_low"].values, rec["ci_95_high"].values - x_rec]
  ax.errorbar(
      x_rec,
      y_idx - offset,
      xerr=err_rec,
      fmt="o",
      color=condition_colors["Recursive"],
      label="Recursive (g0 → g3)",
      capsize=4,
      elinewidth=2,
      markersize=7,
  )

  ctrl = (
      agg[agg["condition"] == "Human_Control"]
      .set_index("metric")
      .loc[metrics]
  )
  x_ctrl = ctrl["mean_change"].values
  err_ctrl = [
      x_ctrl - ctrl["ci_95_low"].values,
      ctrl["ci_95_high"].values - x_ctrl,
  ]
  ax.errorbar(
      x_ctrl,
      y_idx + offset,
      xerr=err_ctrl,
      fmt="s",
      color=condition_colors["Human_Control"],
      label="Human Control (g0 → g3)",
      capsize=4,
      elinewidth=2,
      markersize=6,
  )

  ax.axvline(0, color="black", linestyle="--", linewidth=0.8, alpha=0.7)
  ax.set_yticks(y_idx)
  ax.set_yticklabels(metrics, fontsize=11)
  ax.set_xlabel(
      "Mean Change (g0 → g3) with 95% CI", fontsize=11, fontweight="bold"
  )
  ax.set_title(model, fontsize=13, fontweight="bold")
  ax.legend(loc="upper left", frameon=True)
  ax.set_xlim(-0.16, 0.08)

plt.suptitle(
    "Generational Drift: Recursive Training vs. Human Control Baseline",
    fontsize=14,
    fontweight="bold",
    y=0.98,
)
plt.tight_layout()
plt.savefig(
    PLOTS_DIR / "within_condition_forest_plot.png", dpi=300, bbox_inches="tight"
)
plt.show()

## Net Treatment Effects with Cross-Seed Consistency

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ANALYSIS_DIR = Path.cwd()
PLOTS_DIR = ANALYSIS_DIR / "plots"
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

recursive_color = "#B52B27"       # red
human_control_color = "#014C8C"   # blue

condition_colors = {
    "Recursive": recursive_color,
    "Human_Control": human_control_color
}

rec_colors = {
    42: "#D9534F",
    123: "#8C1B17"
}

ctrl_colors = {
    42: "#0275D8",
    123: "#01325C"
}

df_cond = pd.read_csv("condition_effects_g0_g3.csv")

plt.style.use(
    "seaborn-v0_8-whitegrid"
    if "seaborn-v0_8-whitegrid" in plt.style.available
    else "default"
)

core_metrics = [
    "context_supported_rate",
    "disease_f1",
    "chemical_f1",
    "cosine_similarity",
    "rouge_l",
    "bertscore_f1",
    "context_unsupported_rate",
]
sub = df_cond[df_cond["metric"].isin(core_metrics)].copy()
sub["display_name"] = sub["metric"].apply(lambda x: x.replace("_", " ").title())

pivot_net = sub.pivot(
    index="display_name", columns="model", values="recursive_minus_control"
).sort_values(by="Qwen2.5-3B")
pivot_s42 = sub.pivot(
    index="display_name", columns="model", values="seed42_effect"
).loc[pivot_net.index]
pivot_s123 = sub.pivot(
    index="display_name", columns="model", values="seed123_effect"
).loc[pivot_net.index]

fig, ax = plt.subplots(figsize=(12, 6.5))
y_pos = np.arange(len(pivot_net))
width = 0.35

ax.barh(
    y_pos - width / 2,
    pivot_net["Qwen2.5-0.5B"],
    width,
    label="Qwen2.5-0.5B (Mean Net Effect)",
    color=human_control_color,
    alpha=0.75,
)
ax.barh(
    y_pos + width / 2,
    pivot_net["Qwen2.5-3B"],
    width,
    label="Qwen2.5-3B (Mean Net Effect)",
    color=recursive_color,
    alpha=0.75,
)

ax.scatter(
    pivot_s42["Qwen2.5-0.5B"],
    y_pos - width / 2,
    color=ctrl_colors[42],
    edgecolors="white",
    linewidth=0.7,
    s=55,
    zorder=5,
    label="0.5B Seed 42",
)
ax.scatter(
    pivot_s123["Qwen2.5-0.5B"],
    y_pos - width / 2,
    color=ctrl_colors[123],
    edgecolors="white",
    linewidth=0.7,
    s=55,
    zorder=5,
    label="0.5B Seed 123",
)

ax.scatter(
    pivot_s42["Qwen2.5-3B"],
    y_pos + width / 2,
    color=rec_colors[42],
    edgecolors="white",
    linewidth=0.7,
    s=55,
    zorder=5,
    label="3B Seed 42",
)
ax.scatter(
    pivot_s123["Qwen2.5-3B"],
    y_pos + width / 2,
    color=rec_colors[123],
    edgecolors="white",
    linewidth=0.7,
    s=55,
    zorder=5,
    label="3B Seed 123",
)

ax.axvline(0, color="black", linestyle="--", linewidth=0.8, alpha=0.7)
ax.set_yticks(y_pos)
ax.set_yticklabels(pivot_net.index, fontsize=11)
ax.set_xlabel(
    "Net Treatment Effect: Recursive Δ − Human Control Δ (g0 → g3)",
    fontsize=11,
    fontweight="bold",
)
ax.set_title(
    "Macro Trend: Net Recursive Treatment Effect & Cross-Seed Consistency",
    fontsize=13,
    fontweight="bold",
)
ax.legend(loc="upper left", frameon=True)
ax.set_xlim(-0.42, 0.42)

plt.tight_layout()
plt.savefig(
    PLOTS_DIR / "condition_effects_cross_seed.png", dpi=300, bbox_inches="tight"
)
plt.show()

## Standardized Effect Size (Cohen's $d_z$) Matrix with Wilcoxon Significance

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from matplotlib.colors import LinearSegmentedColormap

ANALYSIS_DIR = Path.cwd()
PLOTS_DIR = ANALYSIS_DIR / "plots"
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

recursive_color = "#B52B27"       # red
human_control_color = "#014C8C"   # blue

condition_colors = {
    "Recursive": recursive_color,
    "Human_Control": human_control_color
}

rec_colors = {
    42: "#D9534F",
    123: "#8C1B17"
}

ctrl_colors = {
    42: "#0275D8",
    123: "#01325C"
}

df_within = pd.read_csv("within_condition_inference_g0_g3.csv")

plt.style.use(
    "seaborn-v0_8-whitegrid"
    if "seaborn-v0_8-whitegrid" in plt.style.available
    else "default"
)

df_within["group"] = (
    df_within["model"]
    + "\n"
    + df_within["condition"].replace({"Human_Control": "Control"})
)

pivot_dz = df_within.groupby(["metric", "group"])["cohens_dz"].mean().unstack()
pivot_p = df_within.groupby(["metric", "group"])["wilcoxon_p"].min().unstack()

col_order = [
    "Qwen2.5-0.5B\nControl",
    "Qwen2.5-0.5B\nRecursive",
    "Qwen2.5-3B\nControl",
    "Qwen2.5-3B\nRecursive",
]
col_order = [c for c in col_order if c in pivot_dz.columns]
pivot_dz = pivot_dz[col_order]
pivot_p = pivot_p[col_order]

metric_order = [
    "Disease F1",
    "Chemical F1",
    "ROUGE-L",
    "BERTScore F1",
    "Cosine similarity",
    "Perplexity",
]
metric_order = [m for m in metric_order if m in pivot_dz.index]
pivot_dz = pivot_dz.loc[metric_order]
pivot_p = pivot_p.loc[metric_order]

annot = pivot_dz.copy().astype(str)
for r in pivot_dz.index:
  for c in pivot_dz.columns:
    val = pivot_dz.loc[r, c]
    p = pivot_p.loc[r, c]
    stars = (
        "***"
        if p < 0.001
        else ("**" if p < 0.01 else ("*" if p < 0.05 else ""))
    )
    annot.loc[r, c] = f"{val:.2f}{stars}"

custom_cmap = LinearSegmentedColormap.from_list(
    "custom_diverging",
    [human_control_color, "#FFFFFF", recursive_color]
)

fig, ax = plt.subplots(figsize=(9, 5.5))
sns.heatmap(
    pivot_dz,
    annot=annot,
    fmt="",
    cmap=custom_cmap,
    center=0,
    cbar_kws={"label": "Cohen's dz"},
    ax=ax,
    linewidths=0.6,
    annot_kws={"fontsize": 10.5, "fontweight": "bold"},
)

ax.set_title(
    "Standardized Effect Sizes (Cohen's dz) & Significance (*p<0.05,"
    " ***p<0.001)",
    fontsize=12,
    fontweight="bold",
)
ax.set_ylabel("")
ax.set_xlabel("")
plt.xticks(rotation=0, fontsize=10.5)
plt.yticks(fontsize=11)
plt.tight_layout()
plt.savefig(
    PLOTS_DIR / "effect_size_heatmap.png", dpi=300, bbox_inches="tight"
)
plt.show()

## Hallucination Inversion (Faithfulness vs. Hallucination Drift)

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ANALYSIS_DIR = Path.cwd()
PLOTS_DIR = ANALYSIS_DIR / "plots"
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

df_cond = pd.read_csv("condition_effects_g0_g3.csv")

plt.style.use(
    "seaborn-v0_8-whitegrid"
    if "seaborn-v0_8-whitegrid" in plt.style.available
    else "default"
)

fig, ax = plt.subplots(figsize=(9, 5))

metrics = ["context_supported_rate", "context_unsupported_rate"]
labels = [
    "Context Supported Rate\n(Factual Faithfulness)",
    "Context Unsupported Rate\n(Hallucination Rate)",
]

x = np.arange(len(labels))
width = 0.20

rec_05 = df_cond[
    (df_cond["model"] == "Qwen2.5-0.5B") & (df_cond["metric"].isin(metrics))
]["recursive_mean_delta"].values
ctrl_05 = df_cond[
    (df_cond["model"] == "Qwen2.5-0.5B") & (df_cond["metric"].isin(metrics))
]["control_mean_delta"].values

rec_3 = df_cond[
    (df_cond["model"] == "Qwen2.5-3B") & (df_cond["metric"].isin(metrics))
]["recursive_mean_delta"].values
ctrl_3 = df_cond[
    (df_cond["model"] == "Qwen2.5-3B") & (df_cond["metric"].isin(metrics))
]["control_mean_delta"].values

ax.bar(
    x - 1.5 * width,
    rec_05,
    width,
    label="0.5B Recursive",
    color="#2b5c8f",
    alpha=0.9,
)
ax.bar(
    x - 0.5 * width,
    ctrl_05,
    width,
    label="0.5B Human Control",
    color="#2b5c8f",
    alpha=0.45,
    hatch="//",
)
ax.bar(
    x + 0.5 * width,
    rec_3,
    width,
    label="3B Recursive",
    color="#c0392b",
    alpha=0.9,
)
ax.bar(
    x + 1.5 * width,
    ctrl_3,
    width,
    label="3B Human Control",
    color="#c0392b",
    alpha=0.45,
    hatch="//",
)

ax.axhline(0, color="black", linestyle="--", linewidth=0.8, alpha=0.7)
ax.set_xticks(x)
ax.set_xticklabels(labels, fontsize=11, fontweight="bold")
ax.set_ylabel("Mean Change (g0 → g3)", fontsize=11, fontweight="bold")
ax.set_title(
    "Hallucination Inversion: Collapse of Contextual Grounding Across Models",
    fontsize=12,
    fontweight="bold",
)
ax.legend(loc="upper left", frameon=True)
ax.set_ylim(-0.30, 0.30)

plt.tight_layout()
plt.savefig(
    PLOTS_DIR / "hallucination_inversion.png", dpi=300, bbox_inches="tight"
)
plt.show()

## Cross-Model Generational Trajectory ($G_0$ to $G_3$)

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ANALYSIS_DIR = Path.cwd()
PLOTS_DIR = ANALYSIS_DIR / "plots"
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

df_05 = pd.read_csv(
    "../evaluation_qwen2.5-0.5b/final_results/data/micro_f1_generation_means.csv"
)
df_3 = pd.read_csv(
    "../evaluation_qwen2.5-3b/final_results/data/micro_f1_generation_means.csv"
)
df_05["model"] = "Qwen2.5-0.5B"
df_3["model"] = "Qwen2.5-3B"
df_traj = pd.concat([df_05, df_3], ignore_index=True)

plt.style.use(
    "seaborn-v0_8-whitegrid"
    if "seaborn-v0_8-whitegrid" in plt.style.available
    else "default"
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5.2), sharey=True)
entity_types = ["disease", "chemical"]
titles = [
    "(A) Disease Extraction Micro F1",
    "(B) Chemical Extraction Micro F1",
]

for ax, etype, title in zip(axes, entity_types, titles):
  sub = df_traj[df_traj["entity_type"] == etype]

  s1 = sub[
      (sub["model"] == "Qwen2.5-0.5B") & (sub["condition"] == "Recursive")
  ]
  ax.plot(
      s1["generation"],
      s1["micro_f1"],
      marker="o",
      linewidth=2.2,
      color="#2b5c8f",
      label="0.5B Recursive",
  )

  s2 = sub[
      (sub["model"] == "Qwen2.5-0.5B") & (sub["condition"] == "Human_Control")
  ]
  ax.plot(
      s2["generation"],
      s2["micro_f1"],
      marker="o",
      linestyle="--",
      linewidth=1.8,
      color="#2b5c8f",
      alpha=0.5,
      label="0.5B Human Control",
  )

  s3 = sub[(sub["model"] == "Qwen2.5-3B") & (sub["condition"] == "Recursive")]
  ax.plot(
      s3["generation"],
      s3["micro_f1"],
      marker="s",
      linewidth=2.2,
      color="#c0392b",
      label="3B Recursive",
  )

  s4 = sub[
      (sub["model"] == "Qwen2.5-3B") & (sub["condition"] == "Human_Control")
  ]
  ax.plot(
      s4["generation"],
      s4["micro_f1"],
      marker="s",
      linestyle="--",
      linewidth=1.8,
      color="#c0392b",
      alpha=0.5,
      label="3B Human Control",
  )

  ax.set_title(title, fontsize=12, fontweight="bold")
  ax.set_xlabel("Generation", fontsize=11, fontweight="bold")
  if ax == axes[0]:
    ax.set_ylabel("Entity Micro F1", fontsize=11, fontweight="bold")
  ax.legend(loc="upper left", frameon=True)

plt.suptitle(
    "Cross-Model Generational Trajectory: Early vs. Late Collapse (G0 to G3)",
    fontsize=14,
    fontweight="bold",
    y=0.99,
)
plt.tight_layout()
plt.savefig(
    PLOTS_DIR / "cross_model_generational_trajectory.png",
    dpi=300,
    bbox_inches="tight",
)
plt.show()

In [ ]:
import pandas as pd

df = pd.read_csv("condition_effects_g0_g3.csv")

# Pivot model/metric into the desired wide format
rows = []

for metric in df["metric"].unique():
    row = {"metric": metric}

    for model in ["Qwen2.5-0.5B", "Qwen2.5-3B"]:
        x = df[(df["model"] == model) & (df["metric"] == metric)].iloc[0]

        prefix = "0.5B" if model.endswith("0.5B") else "3B"

        row[f"{prefix}_recursive"] = x["recursive_mean_delta"]
        row[f"{prefix}_control"] = x["control_mean_delta"]
        row[f"{prefix}_delta_rc"] = x["recursive_minus_control"]

    rows.append(row)

table = pd.DataFrame(rows)
table.to_csv("main_deltas_table.csv", index=False)